# 06 · De bronze a silver (Limpiea)

preparar en `data/silver/` solo lo necesario para estos KPIs, me baso en lo encontrado en los notebooksanteriores:

| Tabla | KPI | Cálculo (se hará en gold) |
|---|---|---|
| ocupaciones | Ingresos de habitaciones | suma de `ingresos_habitaciones` |
| ocupaciones | ADR | suma de ingresos / suma de cuartos noche |
| ocupaciones | ADR sin cuartos sin ingreso | suma de ingresos / (suma de cuartos − suma de cuartos sin ingreso) |
| reservaciones | Cuartos noche por mes de llegada, estado y origen | suma de `cuartos_noche` agrupada |
| reservaciones | Ingresos | suma de `tarifa_total` |

## 0 · Preparación

In [ ]:
import sys; sys.path.append("..")

import pandas as pd
from IPython.display import display

from src.carga import cargar_bronze
from src.rutas import ROOT, SILVER

pd.set_option("display.max_columns", 50)
dfs = cargar_bronze()
{nombre: df.shape for nombre, df in dfs.items()}

In [ ]:
# Valores que vienen de la exploración
VALOR_ANOMALO = 992674.86                    # importe de las 10 filas imposibles
INICIO_VENTANA = pd.Timestamp("2019-10-01")  # desde aquí las llegadas están completas
INICIO_COVID = pd.Timestamp("2020-03-01")    # Inicio de cuarentenas por COVID

# S3: significado hipotético/deducido de cada estatus
ESTADOS = {
    9: "completada",
    2: "cancelada",
    5: "cancelada",
    1: "confirmada_futura",
    7: "alojado",
    8: "alojado",
    3: "sin_determinar",
    4: "sin_determinar",
}

In [ ]:
# Bitácora: aquí se apunta qué hace cada paso y cuántas filas quita
bitacora = []

def anotar(tabla, paso, filas_antes, filas_despues, detalle):
    bitacora.append({
        "tabla": tabla,
        "paso": paso,
        "filas_antes": filas_antes,
        "filas_despues": filas_despues,
        "filas_quitadas": filas_antes - filas_despues,
        "detalle": detalle,
    })
    print(f"[{tabla}] {paso}: {filas_antes:,} → {filas_despues:,} filas ({filas_antes - filas_despues:,} quitadas)")

## 1 · Ocupaciones

### 1.1 · Copia de trabajo

copio la tabla, guardo los totales de bronze para el cuadre final y quito la columna `ADR` original, que está mal en 56.959 filas. El ADR lo recalcularé.

In [ ]:
oc = dfs["ocupaciones"].copy()
ingresos_bronze = oc["ingresos_habitaciones"].sum()

oc = oc.drop(columns=["ADR"])
print(f"Filas: {len(oc):,} | ingresos en bronze: {ingresos_bronze:,.2f}")

### 1.2 · Quitar las filas anómalas

Elimninaré las 10 filas con 200 cuartos y 992.674,86 de ingresos.

Esto porque sumaban casi 10 millones de ingresos imposibles (en 6 de ellas los cuartos superan la capacidad total del hotel) y eso puede afectar la variable ADR.

In [ ]:
es_anomala = (oc["ingresos_habitaciones"] == VALOR_ANOMALO) & (oc["cuartos_noche"] == 200)
ingresos_anomalos = oc.loc[es_anomala, "ingresos_habitaciones"].sum()

antes = len(oc)
oc = oc[~es_anomala]
anotar("ocupaciones", "quitar anómalas", antes, len(oc), "I-04: 200 cuartos y 992.674,86 de ingresos")

### 1.3 · Quitar duplicados exactos

Elimino las filas idénticas en **todas** las columnas, dejando una copia.

In [ ]:
es_duplicada = oc.duplicated()
ingresos_duplicados = oc.loc[es_duplicada, "ingresos_habitaciones"].sum()
print(f"Ingresos en duplicados exactos: {ingresos_duplicados:,.2f} ({ingresos_duplicados / ingresos_bronze:.2%} del total)")

antes = len(oc)
oc = oc[~es_duplicada]
anotar("ocupaciones", "quitar duplicados exactos", antes, len(oc), "Paso 2.2: filas idénticas en todas las columnas")
print("¿Quedan duplicados exactos?", oc.duplicated().any())   # debe dar False

### 1.4 · Quitar las filas sin actividad

Decido eliminar las filas con ingresos = 0 y cuartos = 0.

**Por qué** no cambian ningún KPI (suman 0 a los ingresos y 0 a los cuartos), pero son el 40,8 % de la tabla. Quitarlas hace silver más ligero sin alterar ningún resultado.

In [ ]:
es_vacia = (oc["ingresos_habitaciones"] == 0) & (oc["cuartos_noche"] == 0)

antes = len(oc)
oc = oc[~es_vacia]
anotar("ocupaciones", "quitar filas vacías", antes, len(oc), "I-17: ingresos y cuartos a 0 (no afectan a los KPIs)")

### 1.5 · Columna de cuartos sin ingreso

Crearé `cuartos_sin_ingreso`: vale los cuartos de la fila si tiene cuartos pero ingreso 0, y 0 en las demás.

**Por qué** es lo que necesita el "ADR sin cuartos sin ingreso".

Las filas con ingresos negativos se conservan tal cual: forman parte de los ingresos netos y no son "sin ingreso".

In [ ]:
sin_ingreso = (oc["ingresos_habitaciones"] == 0) & (oc["cuartos_noche"] > 0)
oc["cuartos_sin_ingreso"] = 0
oc.loc[sin_ingreso, "cuartos_sin_ingreso"] = oc["cuartos_noche"]

print(f"Filas con cuartos sin ingreso: {sin_ingreso.sum():,}")
print(f"Cuartos sin ingreso: {oc['cuartos_sin_ingreso'].sum():,} de {oc['cuartos_noche'].sum():,} "
      f"({oc['cuartos_sin_ingreso'].sum() / oc['cuartos_noche'].sum():.1%})")

### 1.6 · solo con las columnas necesarias

**Qué hice:** conservo las tres medidas de los KPIs y las dimensiones del grano de la tabla (fecha, hotel, agencia, segmento y tipo de habitación), con los nombres en minúsculas.

In [ ]:
oc = oc[["Fecha_hoy", "ID_empresa", "ID_Agencia", "ID_Segmento_Mercado", "ID_Tipo_Habitacion",
         "ingresos_habitaciones", "cuartos_noche", "cuartos_sin_ingreso"]]
oc.columns = oc.columns.str.lower()
oc.head()

## 2 · Reservaciones

### 2.1 · Copia de trabajo y fecha de llegada

 convierto `fecha_llegada` de texto (`AAAAMMDD`) a fecha. Las que no se pueden convertir quedan vacías (`NaT`). Solo convertimos esta fecha porque es la única que usan los KPIs (el mes de llegada).

In [ ]:
res = dfs["reservaciones"].copy()
cuartos_bronze = res["num_noches"].sum()
tarifa_bronze = res["tarifa_total"].sum()

res["fecha_llegada"] = pd.to_datetime(res["fecha_llegada"], format="%Y%m%d", errors="coerce")
print(f"Filas: {len(res):,} | llegadas no convertibles: {res['fecha_llegada'].isna().sum():,}")

### 2.2 · Quitar los registros sin data/valores

Me quedaré solo con las reservas que tienen al menos una habitación.

**Por qué:** las que no tienen habitaciones, cuartos noche ni tarifa. **No alterarían ninguno de estos KPIs de sumatorias**. Las quito igualmente para que nadie cuente reservas o calcule medias por reserva sobre una tabla con la mitad de filas vacías.

Este filtro también quita unas 246 reservas reales con 0 habitaciones, que tampoco aportan nada a las sumas.

In [ ]:
sin_habitaciones = res["total_habitaciones"] == 0
cuartos_fantasma = res.loc[sin_habitaciones, "num_noches"].sum()
tarifa_fantasma = res.loc[sin_habitaciones, "tarifa_total"].sum()
print(f"Cuartos noche y tarifa en las filas quitadas: {cuartos_fantasma:,} y {tarifa_fantasma:,.2f} (deberían ser 0)")

antes = len(res)
res = res[~sin_habitaciones]
anotar("reservaciones", "quitar filas sin habitaciones", antes, len(res), "I-12: fantasmas (0 habitaciones, 0 cuartos, 0 tarifa)")

### 2.3 · Quitar reservas sin fecha de llegada

quitaré las pocas reservas cuya fecha de llegada no se pudo convertir.

**Por qué:** no se pueden asignar a ningún mes. Si se quedaran, los cuartos noche por mes no sumarían lo mismo que los cuartos noche por estado u origen. La celda imprime cuánto pesan.

In [ ]:
sin_llegada = res["fecha_llegada"].isna()
cuartos_sin_llegada = res.loc[sin_llegada, "num_noches"].sum()
tarifa_sin_llegada = res.loc[sin_llegada, "tarifa_total"].sum()

antes = len(res)
res = res[~sin_llegada]
anotar("reservaciones", "quitar sin fecha de llegada", antes, len(res),
       f"I-07: no asignables a un mes ({cuartos_sin_llegada:,} cuartos noche)")

** debería revisar la fórmula, recuerdo que eran más

### 2.4 · Las tres dimensiones de los KPIs

**Qué haré:**

- **Estado:** traduzco `estatus_reservacion_id` con el diccionario deducido. Un estatus que no esté en el diccionario queda como `sin_determinar`.
- **Origen:** quito la "E" inicial de `estado_huesped`: `EAGU` → `AGU`.
- **Periodo:** clasifico cada llegada para leer bien los KPIs por mes. Los meses `incompleto` no se comparan con los demás, y los `futura` son cartera que aún podía cancelarse.

In [ ]:
res["estado_reserva"] = res["estatus_reservacion_id"].map(ESTADOS).fillna("sin_determinar")
res["origen_huesped"] = res["estado_huesped"].str.strip().str.removeprefix("E")

fecha_extraccion = pd.to_datetime(res["fecha_registro"], format="%Y%m%d", errors="coerce").max()
print("Fecha de extracción aproximada:", fecha_extraccion.date())

llegada = res["fecha_llegada"]
res["periodo"] = "incompleto"
res.loc[(llegada >= INICIO_VENTANA) & (llegada < INICIO_COVID), "periodo"] = "normal"
res.loc[(llegada >= INICIO_COVID) & (llegada <= fecha_extraccion), "periodo"] = "covid"
res.loc[llegada > fecha_extraccion, "periodo"] = "futura"

display(res["estado_reserva"].value_counts())
display(res["periodo"].value_counts())

### 2.5 · Guardar solo con las columnas necesarias

conservaré el ID, la fecha de llegada, las tres dimensiones y las dos medidas, con nombres claros:

| Antes | Después | Por qué |
|---|---|---|
| `ID_Reserva` | `id_reserva` | convención (I-16) |
| `estatus_reservacion_id` | `id_estatus_reserva` | convención (I-16) |
| `num_noches` | `cuartos_noche` | son noches × habitaciones (I-10) |

**Sobre los ingresos:** las reservas canceladas conservan su tarifa (en torno al 90 % la tiene).

In [ ]:
res = res.rename(columns={
    "ID_Reserva": "id_reserva",
    "estatus_reservacion_id": "id_estatus_reserva",
    "num_noches": "cuartos_noche",
})
res = res[["id_reserva", "fecha_llegada", "periodo", "id_estatus_reserva", "estado_reserva",
           "origen_huesped", "cuartos_noche", "tarifa_total"]]
res.head()

### 2.6 · Catálogo de estados

Un catálogo pequeño con el significado deducido de cada estatus, para que en una capa gold quede claro que no está confirmado.

In [ ]:
estados = pd.DataFrame({
    "id_estatus_reserva": list(ESTADOS.keys()),
    "estado_reserva": list(ESTADOS.values()),
    "origen_del_dato": "deducido (S3)",
}).sort_values("id_estatus_reserva")
estados

## 3 · Validaciones

Cada línea debe imprimir `True`. Si alguna da `False` algo hice mal en la fórmula del notebook pasado

In [ ]:
print("Ocupaciones")
print("  sin filas anómalas:                  ", not ((oc["ingresos_habitaciones"] == VALOR_ANOMALO) & (oc["cuartos_noche"] == 200)).any())
print("  sin filas vacías:                    ", not ((oc["ingresos_habitaciones"] == 0) & (oc["cuartos_noche"] == 0)).any())
print("  cuartos sin ingreso bien calculados: ",
      ((oc["cuartos_sin_ingreso"] == 0) | (oc["cuartos_sin_ingreso"] == oc["cuartos_noche"])).all())

print("\nReservaciones")
print("  id_reserva único:                    ", res["id_reserva"].is_unique)
print("  todas con fecha de llegada:          ", res["fecha_llegada"].notna().all())
print("  cuartos noche no negativos:          ", (res["cuartos_noche"] >= 0).all())
print("  todos los estados traducidos:        ", res["estado_reserva"].notna().all())

## 4 · Guardar silver

primero vacío `data/silver/` y después guardamos las tres tablas. Esto para mantener la idempotencia y no duplicar data

In [ ]:
for archivo in SILVER.glob("*.parquet"):
    archivo.unlink()

tablas_silver = {"ocupaciones": oc, "reservaciones": res, "estados_reserva": estados}
for nombre, df in tablas_silver.items():
    df.to_parquet(SILVER / f"{nombre}.parquet", index=False)
    print(f"{nombre}: {len(df):,} filas")

bitacora_df = pd.DataFrame(bitacora)
bitacora_df.to_csv(ROOT / "reports" / "bitacora_silver.csv", index=False)
bitacora_df

## 5 · Qué queda en silver

| Tabla | Columnas | Para qué KPI |
|---|---|---|
| `ocupaciones` | `fecha_hoy`, `id_empresa`, `id_agencia`, `id_segmento_mercado`, `id_tipo_habitacion`, `ingresos_habitaciones`, `cuartos_noche`, `cuartos_sin_ingreso` | Ingresos, ADR, ADR sin cuartos sin ingreso |
| `reservaciones` | `id_reserva`, `fecha_llegada`, `periodo`, `id_estatus_reserva`, `estado_reserva`, `origen_huesped`, `cuartos_noche`, `tarifa_total` | Cuartos noche por mes, estado y origen; ingresos |
| `estados_reserva` | `id_estatus_reserva`, `estado_reserva`, `origen_del_dato` | Documenta que el significado de los estados es deducido |

**Recordatorios para gold**

- El ADR se calcula **suma entre suma**, nunca promediando.
- `fecha_hoy` no es la fecha de estancia (S1): agrupa por ella, pero llámala por su nombre.
- Los ingresos de reservaciones son los de las **completadas**.
- Los KPIs por mes se leen con el `periodo` al lado.